In [5]:
import pandas as pd
from pathlib import Path

file = Path(
    "../data/raw/primary/"
    "DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv"
)
outpatient = pd.read_csv(file)

print("Shape:", outpatient.shape)

print("\nFirst 5 rows:")
print(outpatient.head())

print("\nColumns:")
print(outpatient.columns.tolist())

C:\Users\Admin\AppData\Local\Temp\ipykernel_11700\3612814786.py:8: DtypeWarning: Columns (21,23,24,25,26,27) have mixed types. Specify dtype option on import or set low_memory=False.
  outpatient = pd.read_csv(file)


Shape: (790790, 76)

First 5 rows:
        DESYNPUF_ID           CLM_ID  SEGMENT  CLM_FROM_DT  CLM_THRU_DT  \
0  00013D2EFD8E45D1  542192281063886        1   20080904.0   20080904.0   
1  00016F745862898F  542272281166593        1   20090602.0   20090602.0   
2  00016F745862898F  542282281644416        1   20090623.0   20090623.0   
3  0001FDD721E223DC  542642281250669        1   20091011.0   20091011.0   
4  00024B3D2352D2D0  542242281386963        1   20080712.0   20080712.0   

  PRVDR_NUM  CLM_PMT_AMT  NCH_PRMRY_PYR_CLM_PD_AMT  AT_PHYSN_NPI  \
0    2600RA         50.0                       0.0  4.824842e+09   
1    3901GS         30.0                       0.0  2.963420e+09   
2    3939PG         30.0                       0.0  5.737808e+09   
3    3902NU         30.0                       0.0  1.233848e+09   
4    5200TV         30.0                       0.0  9.688809e+09   

   OP_PHYSN_NPI  ...  HCPCS_CD_36  HCPCS_CD_37 HCPCS_CD_38 HCPCS_CD_39  \
0           NaN  ...          N

In [6]:
print("\nRows:", len(outpatient))
print("Columns:", len(outpatient.columns))

print("\nData types:")
print(outpatient.dtypes)


Rows: 790790
Columns: 76

Data types:
DESYNPUF_ID     object
CLM_ID           int64
SEGMENT          int64
CLM_FROM_DT    float64
CLM_THRU_DT    float64
                ...   
HCPCS_CD_41     object
HCPCS_CD_42     object
HCPCS_CD_43     object
HCPCS_CD_44     object
HCPCS_CD_45    float64
Length: 76, dtype: object


In [7]:
print("Missing DESYNPUF_ID:",
      outpatient["DESYNPUF_ID"].isna().sum())

print("Missing CLM_ID:",
      outpatient["CLM_ID"].isna().sum())

print("Unique beneficiaries:",
      outpatient["DESYNPUF_ID"].nunique())

print("Unique claims:",
      outpatient["CLM_ID"].nunique())

print("Duplicate CLM_ID:",
      outpatient["CLM_ID"].duplicated().sum())

Missing DESYNPUF_ID: 0
Missing CLM_ID: 0
Unique beneficiaries: 85272
Unique claims: 779815
Duplicate CLM_ID: 10975


In [8]:
print("\nSEGMENT counts:")
print(outpatient["SEGMENT"].value_counts(dropna=False))


SEGMENT counts:
SEGMENT
1    779537
2     11253
Name: count, dtype: int64


In [9]:
# Check whether duplicate CLM_IDs are actually segment 1 + segment 2
duplicate_claims = outpatient[
    outpatient["CLM_ID"].duplicated(keep=False)
].sort_values(["CLM_ID", "SEGMENT"])

print("Rows involved in duplicated CLM_IDs:",
      len(duplicate_claims))

print("\nSegment distribution among duplicate CLM_IDs:")
print(duplicate_claims["SEGMENT"].value_counts())

print("\nNumber of duplicated CLM_ID values:")
print(duplicate_claims["CLM_ID"].nunique())

display(
    duplicate_claims[
        ["CLM_ID", "SEGMENT", "DESYNPUF_ID", "PRVDR_NUM",
         "CLM_PMT_AMT", "CLM_FROM_DT", "CLM_THRU_DT"]
    ].head(30)
)

Rows involved in duplicated CLM_IDs: 21950

Segment distribution among duplicate CLM_IDs:
SEGMENT
1    10975
2    10975
Name: count, dtype: int64

Number of duplicated CLM_ID values:
10975


,CLM_ID,SEGMENT,DESYNPUF_ID,PRVDR_NUM,CLM_PMT_AMT,CLM_FROM_DT,CLM_THRU_DT
730933,542012280839653,1,ECBF343A71C5636E,4200GH,2800.0,20090220.0,20090312.0
730885,542012280839653,2,ECBF343A71C5636E,4213VD,300.0,NaN,NaN
188322,542012280850900,1,3BD1D40B8063F720,3902GN,2800.0,20100514.0,20100603.0
188303,542012280850900,2,3BD1D40B8063F720,3903AG,2300.0,NaN,NaN
699153,542012280851865,1,E254068783BF6B82,3302ND,60.0,20080122.0,20080211.0
699152,542012280851865,2,E254068783BF6B82,3100UB,300.0,NaN,NaN
648918,542012280875173,1,D18895555EB76C37,3302SB,2800.0,20091217.0,20100106.0
648884,542012280875173,2,D18895555EB76C37,1002PP,2400.0,NaN,NaN
272963,542012280876469,1,56F0484C7F9DE78D,3100XU,2400.0,20080726.0,20080815.0
272946,542012280876469,2,56F0484C7F9DE78D,3100XU,2900.0,NaN,NaN


In [10]:
segment_counts = (
    outpatient
    .groupby("CLM_ID")["SEGMENT"]
    .nunique()
)

print("Claims with only one segment:")
print((segment_counts == 1).sum())

print("Claims with multiple segments:")
print((segment_counts > 1).sum())

print("\nSegment combinations:")
print(
    outpatient
    .groupby("CLM_ID")["SEGMENT"]
    .apply(lambda x: tuple(sorted(x.unique())))
    .value_counts()
)

Claims with only one segment:
768840
Claims with multiple segments:
10975

Segment combinations:
SEGMENT
(1,)      768562
(1, 2)     10975
(2,)         278
Name: count, dtype: int64


In [12]:
print("Segment 1 shape:",
      outpatient[outpatient["SEGMENT"] == 1].shape)

print("Segment 2 shape:",
      outpatient[outpatient["SEGMENT"] == 2].shape)

print("\nMissing values in Segment 2:")
print(
    outpatient[outpatient["SEGMENT"] == 2]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(30)
)

Segment 1 shape: (779537, 76)
Segment 2 shape: (11253, 76)

Missing values in Segment 2:
HCPCS_CD_8     11253
HCPCS_CD_27    11253
HCPCS_CD_25    11253
HCPCS_CD_24    11253
HCPCS_CD_23    11253
HCPCS_CD_22    11253
HCPCS_CD_21    11253
HCPCS_CD_20    11253
HCPCS_CD_19    11253
HCPCS_CD_18    11253
HCPCS_CD_17    11253
HCPCS_CD_16    11253
HCPCS_CD_15    11253
HCPCS_CD_14    11253
HCPCS_CD_13    11253
HCPCS_CD_12    11253
HCPCS_CD_11    11253
HCPCS_CD_26    11253
HCPCS_CD_28    11253
HCPCS_CD_9     11253
HCPCS_CD_29    11253
HCPCS_CD_44    11253
HCPCS_CD_43    11253
HCPCS_CD_42    11253
HCPCS_CD_41    11253
HCPCS_CD_40    11253
HCPCS_CD_39    11253
HCPCS_CD_38    11253
HCPCS_CD_37    11253
HCPCS_CD_36    11253
dtype: int64


In [13]:
seg1_ids = set(
    outpatient.loc[outpatient["SEGMENT"] == 1, "CLM_ID"]
)

seg2 = outpatient[outpatient["SEGMENT"] == 2]

seg2["HAS_SEG1_MATCH"] = seg2["CLM_ID"].isin(seg1_ids)

print(
    seg2["HAS_SEG1_MATCH"].value_counts()
)

HAS_SEG1_MATCH
True     10975
False      278
Name: count, dtype: int64


C:\Users\Admin\AppData\Local\Temp\ipykernel_11700\2660649318.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  seg2["HAS_SEG1_MATCH"] = seg2["CLM_ID"].isin(seg1_ids)


In [14]:
date_cols = [
    "CLM_FROM_DT",
    "CLM_THRU_DT"
]

for col in date_cols:
    print(f"\n{col}")
    print("Missing:", outpatient[col].isna().sum())
    print("Unique:", outpatient[col].nunique())
    print("Min:", outpatient[col].min())
    print("Max:", outpatient[col].max())


CLM_FROM_DT
Missing: 11253
Unique: 1116
Min: 20071212.0
Max: 20101231.0

CLM_THRU_DT
Missing: 11253
Unique: 1096
Min: 20080101.0
Max: 20101231.0


In [15]:
seg2_unmatched = outpatient[
    (outpatient["SEGMENT"] == 2) &
    (~outpatient["CLM_ID"].isin(seg1_ids))
].copy()

print("Unmatched Segment 2 rows:", len(seg2_unmatched))

print(
    seg2_unmatched[
        [
            "CLM_ID",
            "DESYNPUF_ID",
            "SEGMENT",
            "PRVDR_NUM",
            "CLM_PMT_AMT",
            "NCH_PRMRY_PYR_CLM_PD_AMT",
            "CLM_FROM_DT",
            "CLM_THRU_DT"
        ]
    ].head(30)
)
print(
    "Unique unmatched CLM_IDs:",
    seg2_unmatched["CLM_ID"].nunique()
)

print(
    "Unique beneficiaries:",
    seg2_unmatched["DESYNPUF_ID"].nunique()
)

print(
    "\nProvider count:",
    seg2_unmatched["PRVDR_NUM"].nunique()
)

print(
    "\nPayment statistics:"
)

print(
    seg2_unmatched["CLM_PMT_AMT"].describe()
)

print(
    "\nSegment 2 unmatched by beneficiary:"
)

print(
    seg2_unmatched["DESYNPUF_ID"].value_counts().describe()
)

Unmatched Segment 2 rows: 278
                 CLM_ID       DESYNPUF_ID  SEGMENT PRVDR_NUM  CLM_PMT_AMT  \
1398    542262281054153  0061141AB18FFA96        2    1401ZD        200.0   
8353    542882281517668  028CC4EDC787721F        2    0101JA        400.0   
8354    542982281015653  028CC4EDC787721F        2    0101JA        200.0   
8499    542462281129179  029C680E4D941320        2    1402JQ         40.0   
12476   542092281181050  03C244F1A64B223A        2    2200NN       1700.0   
12478   542212281331547  03C244F1A64B223A        2    3902VV       2400.0   
12479   542382281486487  03C244F1A64B223A        2    2301GU       3300.0   
12480   542512281074174  03C244F1A64B223A        2    3601YT       2400.0   
12481   542592281422692  03C244F1A64B223A        2    3013PT       1800.0   
13814   542252281069870  042B6E54CC334AAF        2    3418NQ         90.0   
17188   542512281339162  055089D5F99047CA        2    3400ZQ        200.0   
21289   542222280943293  06951796718E2CF2     

In [16]:
# Identify unmatched Segment 2 records
seg2_unmatched = outpatient[
    (outpatient["SEGMENT"] == 2) &
    (~outpatient["CLM_ID"].isin(seg1_ids))
].copy()

# Diagnosis columns
diagnosis_cols = [
    c for c in outpatient.columns
    if "ICD9_DGNS" in c or c == "ADMTNG_ICD9_DGNS_CD"
]

# Procedure columns
procedure_cols = [
    c for c in outpatient.columns
    if "ICD9_PRCDR" in c
]

# HCPCS columns
hcpcs_cols = [
    c for c in outpatient.columns
    if "HCPCS" in c
]

print("Diagnosis columns:", len(diagnosis_cols))
print("Procedure columns:", len(procedure_cols))
print("HCPCS columns:", len(hcpcs_cols))

print("\nNon-missing diagnosis values:")
print(seg2_unmatched[diagnosis_cols].notna().sum())

print("\nNon-missing procedure values:")
print(seg2_unmatched[procedure_cols].notna().sum())

print("\nNon-missing HCPCS values:")
print(seg2_unmatched[hcpcs_cols].notna().sum())

Diagnosis columns: 11
Procedure columns: 6
HCPCS columns: 45

Non-missing diagnosis values:
ICD9_DGNS_CD_1         113
ICD9_DGNS_CD_2           0
ICD9_DGNS_CD_3           0
ICD9_DGNS_CD_4           0
ICD9_DGNS_CD_5           0
ICD9_DGNS_CD_6           0
ICD9_DGNS_CD_7           0
ICD9_DGNS_CD_8           0
ICD9_DGNS_CD_9           0
ICD9_DGNS_CD_10          0
ADMTNG_ICD9_DGNS_CD      0
dtype: int64

Non-missing procedure values:
ICD9_PRCDR_CD_1    0
ICD9_PRCDR_CD_2    0
ICD9_PRCDR_CD_3    0
ICD9_PRCDR_CD_4    0
ICD9_PRCDR_CD_5    0
ICD9_PRCDR_CD_6    0
dtype: int64

Non-missing HCPCS values:
HCPCS_CD_1     227
HCPCS_CD_2       0
HCPCS_CD_3       0
HCPCS_CD_4       0
HCPCS_CD_5       0
HCPCS_CD_6       0
HCPCS_CD_7       0
HCPCS_CD_8       0
HCPCS_CD_9       0
HCPCS_CD_10      0
HCPCS_CD_11      0
HCPCS_CD_12      0
HCPCS_CD_13      0
HCPCS_CD_14      0
HCPCS_CD_15      0
HCPCS_CD_16      0
HCPCS_CD_17      0
HCPCS_CD_18      0
HCPCS_CD_19      0
HCPCS_CD_20      0
HCPCS_CD_21      0
HC

In [17]:
outpatient_features = outpatient.copy()

In [23]:
outpatient_features["CLAIM_KEY"] = (
    outpatient_features["CLM_ID"].astype(str)
    + "_"
    + outpatient_features["SEGMENT"].astype(str)
)
print("Rows:", len(outpatient_features))
print("Unique CLAIM_KEY:", outpatient_features["CLAIM_KEY"].nunique())

Rows: 790790
Unique CLAIM_KEY: 790790


In [ ]:
date_cols = ["CLM_FROM_DT", "CLM_THRU_DT"]

for col in date_cols:
    outpatient_features[col] = pd.to_datetime(
        outpatient_features[col].astype("Int64").astype(str),
        format="%Y%m%d",
        errors="coerce"
    )

In [25]:
print(outpatient_features[date_cols].dtypes)

CLM_FROM_DT    datetime64[ns]
CLM_THRU_DT    datetime64[ns]
dtype: object


In [27]:
outpatient_features["CLAIM_DURATION_DAYS"] = (
    outpatient_features["CLM_THRU_DT"]
    - outpatient_features["CLM_FROM_DT"]
).dt.days + 1

print(
    outpatient_features["CLAIM_DURATION_DAYS"].describe()
)

print(
    "Negative durations:",
    (outpatient_features["CLAIM_DURATION_DAYS"] < 0).sum()
)

count    779537.000000
mean          2.385331
std           4.654968
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          21.000000
Name: CLAIM_DURATION_DAYS, dtype: float64
Negative durations: 0


In [28]:
outpatient_features["CLAIM_YEAR"] = (
    outpatient_features["CLM_FROM_DT"].dt.year
)

outpatient_features["CLAIM_MONTH"] = (
    outpatient_features["CLM_FROM_DT"].dt.month
)

In [30]:
diagnosis_cols = [
    c for c in outpatient_features.columns
    if "ICD9_DGNS_CD" in c
]

outpatient_features["DIAGNOSIS_COUNT"] = (
    outpatient_features[diagnosis_cols]
    .notna()
    .sum(axis=1)
)
print(
    outpatient_features["DIAGNOSIS_COUNT"].describe()
)

count    790790.000000
mean          2.868375
std           2.042041
min           0.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          10.000000
Name: DIAGNOSIS_COUNT, dtype: float64


In [31]:
procedure_cols = [
    c for c in outpatient_features.columns
    if "ICD9_PRCDR_CD" in c
]

outpatient_features["PROCEDURE_COUNT"] = (
    outpatient_features[procedure_cols]
    .notna()
    .sum(axis=1)
)

In [32]:
hcpcs_cols = [
    c for c in outpatient_features.columns
    if "HCPCS_CD" in c
]

outpatient_features["HCPCS_COUNT"] = (
    outpatient_features[hcpcs_cols]
    .notna()
    .sum(axis=1)
)

In [33]:
print(
    outpatient_features["HCPCS_COUNT"].describe()
)

count    790790.000000
mean          4.772871
std           7.202642
min           0.000000
25%           1.000000
50%           2.000000
75%           5.000000
max          44.000000
Name: HCPCS_COUNT, dtype: float64


In [34]:
outpatient_features["HAS_DIAGNOSIS"] = (
    outpatient_features["DIAGNOSIS_COUNT"] > 0
).astype(int)

outpatient_features["HAS_PROCEDURE"] = (
    outpatient_features["PROCEDURE_COUNT"] > 0
).astype(int)

outpatient_features["HAS_HCPCS"] = (
    outpatient_features["HCPCS_COUNT"] > 0
).astype(int)

In [35]:
outpatient_features["HAS_NEGATIVE_PAYMENT"] = (
    outpatient_features["CLM_PMT_AMT"] < 0
).astype(int)

outpatient_features["HAS_PRIMARY_PAYER_PAYMENT"] = (
    outpatient_features["NCH_PRMRY_PYR_CLM_PD_AMT"] > 0
).astype(int)

In [36]:
outpatient_features["TOTAL_REIMBURSEMENT"] = (
    outpatient_features["CLM_PMT_AMT"].fillna(0)
    + outpatient_features["NCH_PRMRY_PYR_CLM_PD_AMT"].fillna(0)
)

In [37]:
outpatient_features["IS_SEGMENT_2"] = (
    outpatient_features["SEGMENT"] == 2
).astype(int)

In [38]:
seg1_ids = set(
    outpatient_features.loc[
        outpatient_features["SEGMENT"] == 1,
        "CLM_ID"
    ]
)

outpatient_features["HAS_SEGMENT_1_MATCH"] = (
    outpatient_features["CLM_ID"].isin(seg1_ids)
).astype(int)

In [39]:
feature_cols = [
    "CLAIM_KEY",
    "DESYNPUF_ID",
    "CLM_ID",
    "SEGMENT",
    "PRVDR_NUM",

    "CLM_PMT_AMT",
    "NCH_PRMRY_PYR_CLM_PD_AMT",
    "NCH_BENE_BLOOD_DDCTBL_LBLTY_AM",
    "NCH_BENE_PTB_DDCTBL_AMT",
    "NCH_BENE_PTB_COINSRNC_AMT",
    "TOTAL_REIMBURSEMENT",

    "CLM_FROM_DT",
    "CLM_THRU_DT",
    "CLAIM_DURATION_DAYS",
    "CLAIM_YEAR",
    "CLAIM_MONTH",

    "DIAGNOSIS_COUNT",
    "PROCEDURE_COUNT",
    "HCPCS_COUNT",

    "HAS_DIAGNOSIS",
    "HAS_PROCEDURE",
    "HAS_HCPCS",

    "HAS_NEGATIVE_PAYMENT",
    "HAS_PRIMARY_PAYER_PAYMENT",

    "IS_SEGMENT_2",
    "HAS_SEGMENT_1_MATCH"
]

claim_features = outpatient_features[feature_cols].copy()

print("Shape:", claim_features.shape)
print("\nColumns:")
print(claim_features.columns.tolist())

Shape: (790790, 26)

Columns:
['CLAIM_KEY', 'DESYNPUF_ID', 'CLM_ID', 'SEGMENT', 'PRVDR_NUM', 'CLM_PMT_AMT', 'NCH_PRMRY_PYR_CLM_PD_AMT', 'NCH_BENE_BLOOD_DDCTBL_LBLTY_AM', 'NCH_BENE_PTB_DDCTBL_AMT', 'NCH_BENE_PTB_COINSRNC_AMT', 'TOTAL_REIMBURSEMENT', 'CLM_FROM_DT', 'CLM_THRU_DT', 'CLAIM_DURATION_DAYS', 'CLAIM_YEAR', 'CLAIM_MONTH', 'DIAGNOSIS_COUNT', 'PROCEDURE_COUNT', 'HCPCS_COUNT', 'HAS_DIAGNOSIS', 'HAS_PROCEDURE', 'HAS_HCPCS', 'HAS_NEGATIVE_PAYMENT', 'HAS_PRIMARY_PAYER_PAYMENT', 'IS_SEGMENT_2', 'HAS_SEGMENT_1_MATCH']


In [40]:
print("Rows:", len(claim_features))

print(
    "Unique CLAIM_KEY:",
    claim_features["CLAIM_KEY"].nunique()
)

print(
    "\nSegment counts:"
)

print(
    claim_features["SEGMENT"].value_counts()
)

print(
    "\nSegment 2 / match:"
)

print(
    claim_features[
        claim_features["SEGMENT"] == 2
    ]["HAS_SEGMENT_1_MATCH"].value_counts()
)

Rows: 790790
Unique CLAIM_KEY: 790790

Segment counts:
SEGMENT
1    779537
2     11253
Name: count, dtype: int64

Segment 2 / match:
HAS_SEGMENT_1_MATCH
1    10975
0      278
Name: count, dtype: int64


In [41]:
numeric_features = [
    "CLM_PMT_AMT",
    "NCH_PRMRY_PYR_CLM_PD_AMT",
    "NCH_BENE_BLOOD_DDCTBL_LBLTY_AM",
    "NCH_BENE_PTB_DDCTBL_AMT",
    "NCH_BENE_PTB_COINSRNC_AMT",
    "TOTAL_REIMBURSEMENT",
    "CLAIM_DURATION_DAYS",
    "DIAGNOSIS_COUNT",
    "PROCEDURE_COUNT",
    "HCPCS_COUNT"
]

print(
    claim_features[numeric_features].describe().T
)

                                   count        mean         std    min   25%  \
CLM_PMT_AMT                     790790.0  283.924569  571.392794 -100.0  40.0   
NCH_PRMRY_PYR_CLM_PD_AMT        790790.0   10.239760  234.668372    0.0   0.0   
NCH_BENE_BLOOD_DDCTBL_LBLTY_AM  790790.0    0.012898    2.315506    0.0   0.0   
NCH_BENE_PTB_DDCTBL_AMT         790790.0    2.825466   15.596522    0.0   0.0   
NCH_BENE_PTB_COINSRNC_AMT       790790.0   83.845876  178.759708    0.0   0.0   
TOTAL_REIMBURSEMENT             790790.0  294.164329  621.547641 -100.0  40.0   
CLAIM_DURATION_DAYS             779537.0    2.385331    4.654968    1.0   1.0   
DIAGNOSIS_COUNT                 790790.0    2.868375    2.042041    0.0   1.0   
PROCEDURE_COUNT                 790790.0    0.000642    0.049064    0.0   0.0   
HCPCS_COUNT                     790790.0    4.772871    7.202642    0.0   1.0   

                                 50%    75%      max  
CLM_PMT_AMT                     80.0  200.0   3300.0 

In [42]:
print(
    "\nMissing values:"
)

print(
    claim_features[numeric_features]
    .isna()
    .sum()
)


Missing values:
CLM_PMT_AMT                           0
NCH_PRMRY_PYR_CLM_PD_AMT              0
NCH_BENE_BLOOD_DDCTBL_LBLTY_AM        0
NCH_BENE_PTB_DDCTBL_AMT               0
NCH_BENE_PTB_COINSRNC_AMT             0
TOTAL_REIMBURSEMENT                   0
CLAIM_DURATION_DAYS               11253
DIAGNOSIS_COUNT                       0
PROCEDURE_COUNT                       0
HCPCS_COUNT                           0
dtype: int64


In [43]:
provider_features = (
    claim_features
    .groupby("PRVDR_NUM")
    .agg(
        CLAIM_COUNT=("CLAIM_KEY", "count"),
        UNIQUE_CLAIM_COUNT=("CLM_ID", "nunique"),
        UNIQUE_BENEFICIARIES=("DESYNPUF_ID", "nunique"),

        TOTAL_PAYMENT=("CLM_PMT_AMT", "sum"),
        AVG_PAYMENT=("CLM_PMT_AMT", "mean"),
        MEDIAN_PAYMENT=("CLM_PMT_AMT", "median"),
        MAX_PAYMENT=("CLM_PMT_AMT", "max"),

        AVG_TOTAL_REIMBURSEMENT=("TOTAL_REIMBURSEMENT", "mean"),
        MAX_TOTAL_REIMBURSEMENT=("TOTAL_REIMBURSEMENT", "max"),

        AVG_CLAIM_DURATION=("CLAIM_DURATION_DAYS", "mean"),
        MAX_CLAIM_DURATION=("CLAIM_DURATION_DAYS", "max"),

        AVG_DIAGNOSIS_COUNT=("DIAGNOSIS_COUNT", "mean"),
        AVG_HCPCS_COUNT=("HCPCS_COUNT", "mean"),
        AVG_PROCEDURE_COUNT=("PROCEDURE_COUNT", "mean"),

        NEGATIVE_PAYMENT_COUNT=("HAS_NEGATIVE_PAYMENT", "sum"),
        PRIMARY_PAYER_PAYMENT_COUNT=("HAS_PRIMARY_PAYER_PAYMENT", "sum"),

        SEGMENT_2_COUNT=("IS_SEGMENT_2", "sum"),
        UNMATCHED_SEGMENT_2_COUNT=(
            "HAS_SEGMENT_1_MATCH",
            lambda x: ((x == 0)).sum()
        )
    )
    .reset_index()
)

In [44]:
provider_features["CLAIMS_PER_BENEFICIARY"] = (
    provider_features["CLAIM_COUNT"]
    / provider_features["UNIQUE_BENEFICIARIES"]
)
provider_features["NEGATIVE_PAYMENT_RATE"] = (
    provider_features["NEGATIVE_PAYMENT_COUNT"]
    / provider_features["CLAIM_COUNT"]
)

In [45]:
provider_features["PRIMARY_PAYER_PAYMENT_RATE"] = (
    provider_features["PRIMARY_PAYER_PAYMENT_COUNT"]
    / provider_features["CLAIM_COUNT"]
)
provider_features["SEGMENT_2_RATE"] = (
    provider_features["SEGMENT_2_COUNT"]
    / provider_features["CLAIM_COUNT"]
)

In [46]:
provider_features["UNMATCHED_SEGMENT_2_RATE"] = (
    provider_features["UNMATCHED_SEGMENT_2_COUNT"]
    / provider_features["CLAIM_COUNT"]
)

In [47]:
payment_std = (
    claim_features
    .groupby("PRVDR_NUM")["CLM_PMT_AMT"]
    .std()
    .rename("PAYMENT_STD")
)

provider_features = provider_features.merge(
    payment_std,
    on="PRVDR_NUM",
    how="left"
)

In [48]:
duration_std = (
    claim_features
    .groupby("PRVDR_NUM")["CLAIM_DURATION_DAYS"]
    .std()
    .rename("CLAIM_DURATION_STD")
)

provider_features = provider_features.merge(
    duration_std,
    on="PRVDR_NUM",
    how="left"
)

In [49]:
print("Provider feature shape:", provider_features.shape)

print("\nColumns:")
print(provider_features.columns.tolist())

print("\nMissing values:")
print(provider_features.isna().sum())

print("\nSummary:")
print(provider_features.describe().T)

Provider feature shape: (6294, 26)

Columns:
['PRVDR_NUM', 'CLAIM_COUNT', 'UNIQUE_CLAIM_COUNT', 'UNIQUE_BENEFICIARIES', 'TOTAL_PAYMENT', 'AVG_PAYMENT', 'MEDIAN_PAYMENT', 'MAX_PAYMENT', 'AVG_TOTAL_REIMBURSEMENT', 'MAX_TOTAL_REIMBURSEMENT', 'AVG_CLAIM_DURATION', 'MAX_CLAIM_DURATION', 'AVG_DIAGNOSIS_COUNT', 'AVG_HCPCS_COUNT', 'AVG_PROCEDURE_COUNT', 'NEGATIVE_PAYMENT_COUNT', 'PRIMARY_PAYER_PAYMENT_COUNT', 'SEGMENT_2_COUNT', 'UNMATCHED_SEGMENT_2_COUNT', 'CLAIMS_PER_BENEFICIARY', 'NEGATIVE_PAYMENT_RATE', 'PRIMARY_PAYER_PAYMENT_RATE', 'SEGMENT_2_RATE', 'UNMATCHED_SEGMENT_2_RATE', 'PAYMENT_STD', 'CLAIM_DURATION_STD']

Missing values:
PRVDR_NUM                        0
CLAIM_COUNT                      0
UNIQUE_CLAIM_COUNT               0
UNIQUE_BENEFICIARIES             0
TOTAL_PAYMENT                    0
AVG_PAYMENT                      0
MEDIAN_PAYMENT                   0
MAX_PAYMENT                      0
AVG_TOTAL_REIMBURSEMENT          0
MAX_TOTAL_REIMBURSEMENT          0
AVG_CLAIM_DURATI

In [50]:
print(
    "\nProviders:",
    provider_features["PRVDR_NUM"].nunique()
)


Providers: 6294


In [55]:
outpatient.to_csv(
    "Outpatient_Claims_Processed.csv",
    index=False
)

In [56]:
import os

file_path = "Outpatient_Claims_Processed.csv"

print("File exists:", os.path.exists(file_path))
print("File size (MB):", round(os.path.getsize(file_path) / (1024 * 1024), 2))
print("Shape:", outpatient.shape)
print(outpatient.columns.tolist())

File exists: True
File size (MB): 155.63
Shape: (790790, 76)
['DESYNPUF_ID', 'CLM_ID', 'SEGMENT', 'CLM_FROM_DT', 'CLM_THRU_DT', 'PRVDR_NUM', 'CLM_PMT_AMT', 'NCH_PRMRY_PYR_CLM_PD_AMT', 'AT_PHYSN_NPI', 'OP_PHYSN_NPI', 'OT_PHYSN_NPI', 'NCH_BENE_BLOOD_DDCTBL_LBLTY_AM', 'ICD9_DGNS_CD_1', 'ICD9_DGNS_CD_2', 'ICD9_DGNS_CD_3', 'ICD9_DGNS_CD_4', 'ICD9_DGNS_CD_5', 'ICD9_DGNS_CD_6', 'ICD9_DGNS_CD_7', 'ICD9_DGNS_CD_8', 'ICD9_DGNS_CD_9', 'ICD9_DGNS_CD_10', 'ICD9_PRCDR_CD_1', 'ICD9_PRCDR_CD_2', 'ICD9_PRCDR_CD_3', 'ICD9_PRCDR_CD_4', 'ICD9_PRCDR_CD_5', 'ICD9_PRCDR_CD_6', 'NCH_BENE_PTB_DDCTBL_AMT', 'NCH_BENE_PTB_COINSRNC_AMT', 'ADMTNG_ICD9_DGNS_CD', 'HCPCS_CD_1', 'HCPCS_CD_2', 'HCPCS_CD_3', 'HCPCS_CD_4', 'HCPCS_CD_5', 'HCPCS_CD_6', 'HCPCS_CD_7', 'HCPCS_CD_8', 'HCPCS_CD_9', 'HCPCS_CD_10', 'HCPCS_CD_11', 'HCPCS_CD_12', 'HCPCS_CD_13', 'HCPCS_CD_14', 'HCPCS_CD_15', 'HCPCS_CD_16', 'HCPCS_CD_17', 'HCPCS_CD_18', 'HCPCS_CD_19', 'HCPCS_CD_20', 'HCPCS_CD_21', 'HCPCS_CD_22', 'HCPCS_CD_23', 'HCPCS_CD_24', 'HCPCS_CD

In [57]:
print("outpatient:", outpatient.shape)

# Show DataFrame variables currently in memory
for name, obj in globals().items():
    if hasattr(obj, "shape") and hasattr(obj, "columns"):
        if obj.shape[0] == 790790:
            print(name, obj.shape)

outpatient: (790790, 76)
outpatient (790790, 76)
outpatient_features (790790, 91)
claim_features (790790, 26)


In [2]:
import pandas as pd
from pathlib import Path

file = Path(
    "../data/processed/primary/"
    "Outpatient_Claims_Processed.csv"
)

outpatient = pd.read_csv(file)
print("Shape:", outpatient.shape)
print("\nColumns:")
print(outpatient.columns.tolist())

C:\Users\Admin\AppData\Local\Temp\ipykernel_34444\394585378.py:9: DtypeWarning: Columns (21,23,24,25,26,27) have mixed types. Specify dtype option on import or set low_memory=False.
  outpatient = pd.read_csv(file)


Shape: (790790, 76)

Columns:
['DESYNPUF_ID', 'CLM_ID', 'SEGMENT', 'CLM_FROM_DT', 'CLM_THRU_DT', 'PRVDR_NUM', 'CLM_PMT_AMT', 'NCH_PRMRY_PYR_CLM_PD_AMT', 'AT_PHYSN_NPI', 'OP_PHYSN_NPI', 'OT_PHYSN_NPI', 'NCH_BENE_BLOOD_DDCTBL_LBLTY_AM', 'ICD9_DGNS_CD_1', 'ICD9_DGNS_CD_2', 'ICD9_DGNS_CD_3', 'ICD9_DGNS_CD_4', 'ICD9_DGNS_CD_5', 'ICD9_DGNS_CD_6', 'ICD9_DGNS_CD_7', 'ICD9_DGNS_CD_8', 'ICD9_DGNS_CD_9', 'ICD9_DGNS_CD_10', 'ICD9_PRCDR_CD_1', 'ICD9_PRCDR_CD_2', 'ICD9_PRCDR_CD_3', 'ICD9_PRCDR_CD_4', 'ICD9_PRCDR_CD_5', 'ICD9_PRCDR_CD_6', 'NCH_BENE_PTB_DDCTBL_AMT', 'NCH_BENE_PTB_COINSRNC_AMT', 'ADMTNG_ICD9_DGNS_CD', 'HCPCS_CD_1', 'HCPCS_CD_2', 'HCPCS_CD_3', 'HCPCS_CD_4', 'HCPCS_CD_5', 'HCPCS_CD_6', 'HCPCS_CD_7', 'HCPCS_CD_8', 'HCPCS_CD_9', 'HCPCS_CD_10', 'HCPCS_CD_11', 'HCPCS_CD_12', 'HCPCS_CD_13', 'HCPCS_CD_14', 'HCPCS_CD_15', 'HCPCS_CD_16', 'HCPCS_CD_17', 'HCPCS_CD_18', 'HCPCS_CD_19', 'HCPCS_CD_20', 'HCPCS_CD_21', 'HCPCS_CD_22', 'HCPCS_CD_23', 'HCPCS_CD_24', 'HCPCS_CD_25', 'HCPCS_CD_26', 'HCPCS_CD_